# ProteomeLM: gene-essentiality prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bitbol-Lab/ProteomeLM/blob/main/notebooks/essentiality_prediction.ipynb)

Predict which genes of an organism are essential from its proteome alone, with **ProteomeLM-ess** ([PNAS 2026](https://www.pnas.org/doi/10.1073/pnas.2524201123), Fig. 5). Every protein is embedded with ESM-C 600M, [ProteomeLM-L](https://huggingface.co/Bitbol-Lab/ProteomeLM-L) reads the whole proteome at once (one token per protein), and a small classifier on its layer-8 hidden states, trained on the [OGEE](https://v3.ogee.info) essentiality data of 82 genomes, gives each protein a probability of being essential.

**How to use**

1. **Setup**: run the first code cell. On Colab, first pick a GPU runtime (*Runtime → Change runtime type → T4 GPU*); the cell installs ProteomeLM (1–3 min).
2. **Settings**: choose the proteome (a UniProt reference proteome, a UniProt proteome id, a STRING organism or your own FASTA file) and how many proteins to call essential. Optionally, paste known essential genes to compare with.
3. **Run**: *Runtime → Run all*. The results are shown as a table and figures and saved as a TSV file plus PNG/SVG figures (downloaded automatically on Colab).

The default settings score the ~4,400 proteins of *E. coli* K-12 in about a minute on a GPU. The first run also downloads the ESM-C 600M weights (2.3 GB), ProteomeLM-L (0.7 GB) and the classifier (5 MB).

**Use a whole proteome.** ProteomeLM scores each protein in the context of all the others, so give it one complete proteome (one genome), not a hand-picked subset.

In [ ]:
#@title 1. Setup: install (Colab), check the GPU, import { display-mode: "form" }
PROTEOMELM_REPO_URL = "https://github.com/Bitbol-Lab/ProteomeLM"  #@param {type:"string"}
PROTEOMELM_REF = "main"  #@param {type:"string"}

import importlib
import importlib.util
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)  # before torch is imported
warnings.filterwarnings("ignore", message=".*IProgress not found.*")  # tqdm without ipywidgets
IN_COLAB = "google.colab" in sys.modules


def _find_local_clone():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "proteomelm" / "__init__.py").is_file() and (folder / "pyproject.toml").is_file():
            return folder
    return None


LOCAL_CLONE = _find_local_clone()
if LOCAL_CLONE is not None:
    sys.path.insert(0, str(LOCAL_CLONE))  # use the code of the clone this notebook lives in
elif importlib.util.find_spec("proteomelm") is None:
    package = f"proteomelm[colab] @ git+{PROTEOMELM_REPO_URL}@{PROTEOMELM_REF}"
    if not IN_COLAB:
        raise ImportError(f'ProteomeLM is not installed. Run: pip install "{package}"')
    print("Installing ProteomeLM and its dependencies (1-3 min)...")
    install = subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], capture_output=True, text=True)
    if install.returncode != 0:
        print(install.stdout[-3000:], install.stderr[-3000:])
        raise RuntimeError("pip install failed (see the log above).")
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from proteomelm.essentiality import EssentialityPredictor, check_proteome, load_head
from proteomelm.essentiality_notebook import (
    RULES, SOURCES, annotate, build_form, cached_esmc, calling_rule, known_labels, label_metrics, load_proteome,
    match_proteins, offer_downloads, plan_devices, plot_call_donut, plot_label_curves, plot_score_distribution,
    results_stem, save_table,
)
from proteomelm.ppi.notebook_plots import show_figure
from proteomelm.ppi.notebook_runtime import configure_notebook_logging, get_cache_dir, upload_fasta_in_colab

try:
    import ipywidgets  # noqa: F401  (optional: interactive settings form)
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

configure_notebook_logging()
torch.set_grad_enabled(False)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

import proteomelm
print(f"ProteomeLM code: {Path(proteomelm.__file__).parent}")
print(f"Work directory (downloads, embeddings, results): {get_cache_dir()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({gpu.total_memory / 1e9:.0f} GB)")
else:
    print("No GPU found: embedding a whole proteome with ESM-C on CPU is slow (hours for thousands of proteins)."
          + (" On Colab: Runtime > Change runtime type > T4 GPU, then run all cells again." if IN_COLAB else ""))

## 2. Settings

Edit the values below (on Colab, use the form on the right).

- **Proteome**: `UniProt reference proteome` takes an NCBI taxon id (83333 = *E. coli* K-12, 559292 = *S. cerevisiae* S288C, 224308 = *B. subtilis* 168) and downloads its UniProt reference proteome; `UniProt proteome id` takes an id such as `UP000000625`; `STRING organism` takes a STRING taxon id (511145 = *E. coli* K-12); `FASTA file` takes a path (on Colab, leave it empty to upload a file).
- **Calling rule**: which proteins to call essential. The ranking is the reliable part: the probabilities are not calibrated across organisms (the fraction of essential genes ranges from a few percent in free-living bacteria to about half in minimal cells), so `Top fraction` or `Top N` with a number that suits your organism is recommended. `Probability threshold` calls p_essential ≥ threshold.
- **Known essential genes** (optional): gene names, UniProt accessions or protein ids (comma- or space-separated, or the path of a text file with one per line). The notebook then reports AUROC / AUPR and plots ROC and precision-recall curves. Without a non-essential list, every other protein counts as non-essential.
- **Genes to look up**: shown in a separate table.
- **Advanced**: the classifier (a Hugging Face repository or a local folder), devices (`auto` = GPU if available) and the output folder.

In [ ]:
#@title 2. Settings { display-mode: "form" }
#@markdown ### Proteome
proteome_source = "UniProt reference proteome (taxon id)"  #@param ["UniProt reference proteome (taxon id)", "UniProt proteome id (UP...)", "STRING organism (taxon id)", "FASTA file"]
#@markdown Taxon id, UniProt proteome id or FASTA path, depending on the source.
proteome = "83333"  #@param {type:"string"}

#@markdown ### Calling rule
rule = "Top fraction"  #@param ["Top fraction", "Top N", "Probability threshold"]
top_fraction = 0.1  #@param {type:"number"}
top_n = 300  #@param {type:"integer"}
threshold = 0.5  #@param {type:"number"}

#@markdown ### Compare with known essential genes (optional)
known_essential = ""  #@param {type:"string"}
known_nonessential = ""  #@param {type:"string"}

#@markdown ### Genes to look up
genes_to_show = "rpoB, ftsZ, secY, lacZ, recA"  #@param {type:"string"}

#@markdown ### Advanced
head = "Bitbol-Lab/ProteomeLM-ess"  #@param {type:"string"}
device = "auto"  #@param {type:"string"}
esm_device = "auto"  #@param {type:"string"}
#@markdown Output folder (empty = `results/` in the work directory printed by the setup cell).
output_dir = ""  #@param {type:"string"}

CONFIG = {
    "proteome_source": proteome_source, "proteome": proteome,
    "rule": rule, "top_fraction": top_fraction, "top_n": top_n, "threshold": threshold,
    "known_essential": known_essential, "known_nonessential": known_nonessential, "genes_to_show": genes_to_show,
    "head": head, "device": device, "esm_device": esm_device, "output_dir": output_dir,
}

# Optional interactive form (Jupyter/VS Code with ipywidgets). It edits CONFIG in place.
if HAS_WIDGETS and not IN_COLAB:
    display(build_form(CONFIG, {"proteome_source": list(SOURCES), "rule": list(RULES)}))

## 3. Run

### Load the proteome and the classifier
Downloads (or reads) the proteome and the classifier, and picks the devices: ProteomeLM runs on the CPU when the GPU is too small. Downloads are cached in the work directory.

In [ ]:
start = time.time()
if SOURCES[CONFIG["proteome_source"]] == "fasta" and not str(CONFIG["proteome"]).strip():
    if not IN_COLAB:
        raise ValueError("Set `proteome` to the path of a FASTA file.")
    CONFIG["proteome"] = upload_fasta_in_colab()
PROTEOME = load_proteome(CONFIG["proteome_source"], CONFIG["proteome"])
check_proteome(PROTEOME.ids, PROTEOME.sequences)
print(f"Loaded {PROTEOME.n_proteins:,} proteins of {PROTEOME.organism} ({time.time() - start:.0f} s).")

HEAD_CONFIG, HEAD = load_head(CONFIG["head"])  # fails here, before the slow steps, if the classifier is unavailable
print(f"Classifier: {CONFIG['head']} on {HEAD_CONFIG.backbone} hidden_states[{HEAD_CONFIG.layer}]")
PLM_DEVICE, ESM_DEVICE, NOTES = plan_devices(PROTEOME.n_proteins, HEAD_CONFIG.backbone, CONFIG["device"], CONFIG["esm_device"])
for note in NOTES:
    print("Note:", note)
PREDICTOR = EssentialityPredictor(HEAD_CONFIG, HEAD.to(PLM_DEVICE), device=PLM_DEVICE, esm_device=ESM_DEVICE)
RULE = calling_rule(CONFIG["rule"], CONFIG["top_fraction"], CONFIG["top_n"], CONFIG["threshold"])

### Embed the proteome and predict
ESM-C embeds every protein (the slow step, cached for the next run), then ProteomeLM-L reads the whole proteome and the classifier scores each protein.

In [ ]:
start = time.time()
ESMC = cached_esmc(PREDICTOR, PROTEOME)
PREDICTOR.unload_esmc()  # leave the GPU memory to ProteomeLM
esmc_seconds = time.time() - start
TABLE = annotate(PREDICTOR.predict(PROTEOME.ids, PROTEOME.sequences, esmc=ESMC, **RULE), PROTEOME)
print(f"ESM-C: {esmc_seconds:.0f} s on {ESM_DEVICE}; ProteomeLM + classifier: {time.time() - start - esmc_seconds:.0f} s on {PLM_DEVICE}.")
n_called = int(TABLE["predicted_essential"].sum())
print(f"{n_called:,} of {len(TABLE):,} proteins predicted essential ({n_called / len(TABLE):.1%}; rule: {CONFIG['rule']} = {next(iter(RULE.values()))}).")

### Results
Proteins ranked from most to least likely essential, then the genes to look up.

In [ ]:
COLUMNS = ["rank", "gene", "protein_id", "p_essential", "predicted_essential", "description"]
display(TABLE[COLUMNS].head(25).set_index("rank"))

rows, missing = match_proteins(TABLE, CONFIG["genes_to_show"])
if missing:
    print("Not found:", ", ".join(missing))
if rows:
    print("Genes to look up:")
    display(TABLE.iloc[sorted(rows)][COLUMNS].set_index("rank"))

In [ ]:
LABELS, missing = known_labels(TABLE, CONFIG["known_essential"], CONFIG["known_nonessential"])
if missing:
    print(f"{len(missing)} known gene(s) not found in the proteome: {', '.join(missing[:20])}")
if LABELS is None:
    print("No known essential genes given: skipping the comparison.")
else:
    TABLE["known_label"] = pd.Series(LABELS).map({1: "essential", 0: "non-essential", -1: ""}).to_numpy()
    m = label_metrics(TABLE, LABELS)
    print(f"Known labels: {m['n_essential']:,} essential, {m['n_nonessential']:,} non-essential.")
    if "auroc" in m:
        print(f"AUROC {m['auroc']:.3f}; AUPR {m['aupr']:.3f} (random: {m['aupr_baseline']:.3f}); "
              f"{m['essential_in_top_n']} of the {m['n_essential']} known essential genes are in the top {m['n_essential']}.")

TSV = save_table(TABLE, results_stem(PROTEOME.organism), CONFIG["output_dir"] or None)
print(f"Saved {TSV} ({TSV.stat().st_size / 1e6:.1f} MB)")
FIGURE_PREFIX = TSV.with_suffix("")  # figures are saved next to the results (PNG + SVG)
FIGURES = []

## 4. Figures

Each figure is shown here and saved next to the results as PNG and SVG.

### Score distribution
Number of proteins per p_essential bin (log scale), with the proteins called essential in blue and the resulting cutoff.

In [ ]:
FIGURES += show_figure(plot_score_distribution(TABLE, title=f"{PROTEOME.organism}: essentiality scores"), "scores", FIGURE_PREFIX)

### Predicted essential proteins
With known essential genes, the outer ring shows the labels and the inner ring the predictions, one slice per protein, as in Fig. 5B of the paper.

In [ ]:
FIGURES += show_figure(plot_call_donut(TABLE, LABELS), "donut", FIGURE_PREFIX)

### Comparison with known essential genes
ROC and precision-recall curves of p_essential against the known labels (skipped without them). The dotted lines show a random ranking.

In [ ]:
FIGURES += show_figure(plot_label_curves(TABLE, LABELS), "roc_pr", FIGURE_PREFIX)

### Download (Colab)
On Colab this downloads the TSV file and a zip of the figures; elsewhere it prints the folder they are in.

In [ ]:
offer_downloads([TSV], FIGURES)

## Output columns

| Column | Meaning |
|---|---|
| `protein_id` | FASTA record id (e.g. UniProt `sp\|P0A8V2\|RPOB_ECOLI`, STRING `511145.b3987`) |
| `gene`, `description` | gene name and protein name from the FASTA headers (UniProt) or STRING, when available |
| `p_essential` | probability of the essential class. Use it to rank proteins; it is not calibrated across organisms |
| `rank` | 1 = most likely essential |
| `predicted_essential` | the call of the chosen rule (top fraction, top N or threshold) |
| `known_label` | your known labels, when given |

Files: `essentiality_<organism>.tsv` (all proteins, most essential first) and `essentiality_<organism>_<figure>.png` / `.svg` (zipped as `essentiality_<organism>_figures.zip` for download). The same prediction runs from the command line: `python -m proteomelm.essentiality --fasta proteome.fasta --top-fraction 0.1`.

**Citation.** Malbranke, Zalaffi & Bitbol, *ProteomeLM: A proteome-scale language model enables accurate and rapid prediction of protein-protein interactions and gene essentiality across taxa*, PNAS 123 (21), e2524201123 (2026). [doi:10.1073/pnas.2524201123](https://www.pnas.org/doi/10.1073/pnas.2524201123)